# 14D — Factor-controlled portfolio attribution

Do the frozen text, embedding, hybrid, and consensus portfolios merely load on conventional equity risks and the independent critical-minerals theme factor, or is a residual intercept left? This attribution stage never reselects securities or changes weights. It uses daily CRSP returns over the exact frozen three-month windows, with the concatenated calendar-time series as the primary test and each vintage as a stability diagnostic.

## 1. Setup and frozen design

In [1]:
from pathlib import Path
import hashlib,io,json,urllib.request,zipfile
import numpy as np,pandas as pd,statsmodels.api as sm
from IPython.display import display
ROOT=Path.cwd()
if ROOT.name!='sp500_critical_minerals':
 q=Path('/Users/harmandeep/Desktop/Projects/Asset_Embeddings/geopolitical_exposure/sp500_critical_minerals')
 if q.exists():ROOT=q
OUT=ROOT/'factor_attribution_v1';RAW=OUT/'raw';MAN=OUT/'manifests'
for q in [OUT,RAW,MAN]:q.mkdir(exist_ok=True)
DESIGN={'version':'factor_controlled_portfolio_attribution_v1','frequency':'daily','windows':'exact frozen three-month forward windows','primary_panel':'concatenated calendar time by method','secondary_panel':'each vintage','long_short_y':'raw self-financing return','long_only_y':'return minus RF','controls':'daily FF5 plus momentum plus frozen 14A theme factor','HAC_lags':5,'no_reselection':True}
dp=MAN/'attribution_design.json'
if dp.exists():assert json.loads(dp.read_text())==DESIGN
else:dp.write_text(json.dumps(DESIGN,indent=2,sort_keys=True)+'\n')
dh=hashlib.sha256(dp.read_bytes()).hexdigest();print({'design_sha256':dh,'regressions_run':False})

{'design_sha256': '4f9b16bdb335c319e9bbce952a614c9262744a082c42cd411763950497512481', 'regressions_run': False}


## 2. Load frozen memberships, windows, returns, and thematic factors

In [2]:
mp=ROOT/'long_short_hybrid_v1/frozen_portfolio_membership.parquet';cp=ROOT/'consensus_agreement_v1/frozen_consensus_membership.parquet';rp=ROOT/'thematic_beta_v1/crsp_daily_firm_returns.parquet';fp=ROOT/'thematic_factor_v1/daily_critical_minerals_factors.parquet';sp=ROOT/'config/sp500_experiment_design_v1.json';paths=[mp,cp,rp,fp,sp]
audit=pd.DataFrame([{'path':str(q),'exists':q.exists(),'sha256':hashlib.sha256(q.read_bytes()).hexdigest() if q.exists() else None} for q in paths]);display(audit);assert audit.exists.all()
m=pd.concat([pd.read_parquet(mp)[['vintage','method','leg','permco','weight']],pd.read_parquet(cp)[['vintage','method','leg','permco','weight']]],ignore_index=True);m['permco']=pd.to_numeric(m.permco,errors='coerce').astype('Int64');m=m.groupby(['vintage','method','leg','permco'],as_index=False).weight.sum()
d=pd.read_parquet(rp);d['date']=pd.to_datetime(d.date);d['permco']=pd.to_numeric(d.permco,errors='coerce').astype('Int64');d['firm_return']=pd.to_numeric(d.firm_return,errors='coerce');d=d.dropna(subset=['permco','date','firm_return']).drop_duplicates(['permco','date'])
f=pd.read_parquet(fp);f['date']=pd.to_datetime(f.date);cfg=json.loads(sp.read_text());w=pd.DataFrame([{'vintage':v['label'],'month':pd.Period(x,'M')} for v in cfg['vintages'] for x in v['forward_months']]);display(m.groupby(['vintage','method']).agg(names=('permco','nunique'),net=('weight','sum'),gross=('weight',lambda x:x.abs().sum())).reset_index())

,path,exists,sha256
0,/Users/harmandeep/Desktop/Projects/Asset_Embed...,True,99827e60d5a7c86e962bac9d2bd35952c837255e1f6c1a...
1,/Users/harmandeep/Desktop/Projects/Asset_Embed...,True,1a5e114173dc2c140b3672a2bd9f068bdbc60c05270daf...
2,/Users/harmandeep/Desktop/Projects/Asset_Embed...,True,947695ca321f5efb2d0021161575155bc69bdc103b95c6...
3,/Users/harmandeep/Desktop/Projects/Asset_Embed...,True,25d1ddb36a98ae97a0f1340d23893e94dfae2fa68189e4...
4,/Users/harmandeep/Desktop/Projects/Asset_Embed...,True,1c252e3f790722e8822ba9ef2a89b07d9aa14b3bb96e1f...


,vintage,method,names,net,gross
0,2024q1,consensus,27,-2.081668e-17,2.0
1,2024q1,embedding,45,4.163336e-17,2.0
2,2024q1,hybrid,45,4.163336e-17,2.0
3,2024q1,hybrid_long_only,22,1.000000e+00,1.0
4,2024q1,llm,45,4.163336e-17,2.0
5,2024q1,matched_text,27,-2.081668e-17,2.0
6,2024q2,consensus,27,0.000000e+00,2.0
7,2024q2,embedding,42,5.551115e-17,2.0
8,2024q2,hybrid,42,5.551115e-17,2.0
9,2024q2,hybrid_long_only,20,1.000000e+00,1.0


## 3. Reconstruct daily frozen returns

Missing returns contribute zero to the raw calculation but are not hidden: observed gross-weight coverage is audited and no surviving firm is upweighted.

In [3]:
dates=d[['date']].drop_duplicates();dates['month']=dates.date.dt.to_period('M');dates=dates.merge(w,on='month');g=m.merge(dates,on='vintage').merge(d,on=['permco','date'],how='left',validate='many_to_one');g['contribution']=g.weight*g.firm_return;g['absolute']=g.weight.abs();g['observed']=np.where(g.firm_return.notna(),g.absolute,0)
daily=g.groupby(['vintage','method','date'],as_index=False).agg(portfolio_return=('contribution','sum'),target_gross=('absolute','sum'),observed_gross=('observed','sum'),names=('permco','nunique'),observed_names=('firm_return','count'));daily['coverage']=daily.observed_gross/daily.target_gross;coverage=daily.groupby(['vintage','method'],as_index=False).agg(days=('date','nunique'),minimum_coverage=('coverage','min'),mean_coverage=('coverage','mean'));display(coverage);assert coverage.days.min()>=55;assert coverage.minimum_coverage.min()>=.90;daily.to_parquet(OUT/'daily_frozen_portfolio_returns.parquet',index=False)

,vintage,method,days,minimum_coverage,mean_coverage
0,2024q1,consensus,63,1.0,1.0
1,2024q1,embedding,63,1.0,1.0
2,2024q1,hybrid,63,1.0,1.0
3,2024q1,hybrid_long_only,63,1.0,1.0
4,2024q1,llm,63,1.0,1.0
5,2024q1,matched_text,63,1.0,1.0
6,2024q2,consensus,63,1.0,1.0
7,2024q2,embedding,63,1.0,1.0
8,2024q2,hybrid,63,1.0,1.0
9,2024q2,hybrid_long_only,63,1.0,1.0


## 4. Official daily Fama–French controls

Small ZIP files from the [Kenneth French Data Library](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html) are checkpointed locally.

In [4]:
URL={'ff5':'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_5_Factors_2x3_daily_CSV.zip','mom':'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Momentum_Factor_daily_CSV.zip'}
def get(k):
 p=RAW/f'{k}_daily.zip'
 if not p.exists():p.write_bytes(urllib.request.urlopen(urllib.request.Request(URL[k],headers={'User-Agent':'Academic research'}),timeout=60).read())
 return p
def parse(p,needle):
 with zipfile.ZipFile(p) as z:s=z.read([a for a in z.namelist() if a.lower().endswith('.csv')][0]).decode('utf-8-sig',errors='replace').splitlines()
 h=next(i for i,a in enumerate(s) if needle in a.replace(' ',''));rows=[]
 for a in s[h+1:]:
  k=a.split(',')[0].strip()
  if not(len(k)==8 and k.isdigit()):break
  rows.append(a)
 x=pd.read_csv(io.StringIO(s[h]+'\n'+'\n'.join(rows))).rename(columns=lambda a:str(a).strip());x=x.rename(columns={x.columns[0]:'yyyymmdd'});x['date']=pd.to_datetime(x.yyyymmdd.astype(str),format='%Y%m%d');return x.drop(columns='yyyymmdd')
std=parse(get('ff5'),'Mkt-RF').merge(parse(get('mom'),'Mom'),on='date');std.columns=[str(a).strip() for a in std.columns]
for a in ['Mkt-RF','SMB','HML','RMW','CMA','RF','Mom']:std[a]=pd.to_numeric(std[a],errors='coerce')/100
std=std[['date','Mkt-RF','SMB','HML','RMW','CMA','Mom','RF']];std.to_parquet(OUT/'daily_standard_factors.parquet',index=False);display(std.tail())

,date,Mkt-RF,SMB,HML,RMW,CMA,Mom,RF
15849,2026-06-24,-0.0007,0.0078,-0.0021,0.0042,0.0061,-0.0103,0.0001
15850,2026-06-25,-0.0013,0.0061,0.0091,-0.0076,0.0051,0.0162,0.0001
15851,2026-06-26,0.0015,0.0136,-0.0095,0.0044,0.0004,-0.0318,0.0001
15852,2026-06-29,0.0120,-0.0089,-0.0090,-0.0177,-0.0050,0.0166,0.0001
15853,2026-06-30,0.0073,-0.0010,-0.0062,-0.0110,-0.0049,0.0106,0.0001


## 5. Assemble vintage and primary calendar-time panels

In [5]:
panel=daily.merge(std,on='date',validate='many_to_one').merge(f,on='date',validate='many_to_one');panel['dependent_return']=np.where(panel.method.eq('hybrid_long_only'),panel.portfolio_return-panel.RF,panel.portfolio_return);panel.to_parquet(OUT/'daily_attribution_panel.parquet',index=False)
fc=['Mkt-RF','SMB','HML','RMW','CMA','Mom','RF','broad_cm_equity_ex_market','producer_ex_market','base_metals_raw','materials_ex_market'];cal=panel.groupby(['method','date'],as_index=False).agg(portfolio_return=('portfolio_return','mean'),dependent_return=('dependent_return','mean'),live_vintages=('vintage','nunique'),**{a:(a,'first') for a in fc});cal.to_parquet(OUT/'calendar_time_attribution_panel.parquet',index=False);display(cal.groupby('method').agg(days=('date','nunique'),mean_return=('portfolio_return','mean')).reset_index())

,method,days,mean_return
0,consensus,248,-0.000387
1,embedding,248,-0.000861
2,hybrid,248,-0.000916
3,hybrid_long_only,248,-0.000257
4,llm,248,-0.001132
5,matched_text,248,-0.000755


## 6. Frozen attribution regressions

Daily alpha is annualized by 252. HAC standard errors allow short-run dependence. A remaining alpha is not proof of skill because the history is short and transaction costs are absent.

In [6]:
S={'mean_only':[],'capm':['Mkt-RF'],'capm_plus_cm':['Mkt-RF','broad_cm_equity_ex_market'],'ff5':['Mkt-RF','SMB','HML','RMW','CMA'],'ff5_mom':['Mkt-RF','SMB','HML','RMW','CMA','Mom'],'ff5_mom_cm':['Mkt-RF','SMB','HML','RMW','CMA','Mom','broad_cm_equity_ex_market']}
def fit(z,pn,v,m):
 out=[]
 for spec,cols in S.items():
  a=z[['dependent_return']+cols].dropna();r=sm.OLS(a.dependent_return,sm.add_constant(a[cols],has_constant='add')).fit(cov_type='HAC',cov_kwds={'maxlags':5,'use_correction':True});ci=r.conf_int().loc['const'];out.append({'panel':pn,'vintage':v,'method':m,'specification':spec,'days':len(a),'daily_alpha':r.params['const'],'annualized_alpha':252*r.params['const'],'alpha_ci_low':252*ci.iloc[0],'alpha_ci_high':252*ci.iloc[1],'alpha_hac_t':r.tvalues['const'],'alpha_p_value':r.pvalues['const'],'r2':r.rsquared,'cm_beta':r.params.get('broad_cm_equity_ex_market',np.nan),'cm_beta_hac_t':r.tvalues.get('broad_cm_equity_ex_market',np.nan)})
 return out
rows=[]
for (v,m),a in panel.groupby(['vintage','method']):rows+=fit(a,'vintage',v,m)
for m,a in cal.groupby('method'):rows+=fit(a,'calendar_time','all_windows',m)
res=pd.DataFrame(rows);res.to_parquet(OUT/'factor_attribution_results.parquet',index=False);display(res[(res.panel.eq('calendar_time'))&res.specification.isin(['mean_only','ff5_mom','ff5_mom_cm'])][['method','specification','days','annualized_alpha','alpha_ci_low','alpha_ci_high','alpha_hac_t','alpha_p_value','cm_beta','cm_beta_hac_t','r2']].round(4))

,method,specification,days,annualized_alpha,alpha_ci_low,alpha_ci_high,alpha_hac_t,alpha_p_value,cm_beta,cm_beta_hac_t,r2
144,consensus,mean_only,248,-0.0976,-0.6049,0.4097,-0.3770,0.7062,NaN,NaN,0.0000
148,consensus,ff5_mom,248,-0.1966,-0.6406,0.2475,-0.8676,0.3856,NaN,NaN,0.3119
149,consensus,ff5_mom_cm,248,-0.1019,-0.4626,0.2589,-0.5534,0.5800,0.4848,9.6169,0.5144
150,embedding,mean_only,248,-0.2170,-0.4207,-0.0132,-2.0872,0.0369,NaN,NaN,0.0000
154,embedding,ff5_mom,248,-0.2199,-0.4324,-0.0073,-2.0275,0.0426,NaN,NaN,0.1174
155,embedding,ff5_mom_cm,248,-0.2157,-0.4270,-0.0044,-2.0005,0.0454,0.0214,0.9054,0.1202
156,hybrid,mean_only,248,-0.2309,-0.4732,0.0115,-1.8671,0.0619,NaN,NaN,0.0000
160,hybrid,ff5_mom,248,-0.2244,-0.4533,0.0045,-1.9214,0.0547,NaN,NaN,0.1073
161,hybrid,ff5_mom_cm,248,-0.1825,-0.3751,0.0100,-1.8579,0.0632,0.2143,8.5325,0.3125
162,hybrid_long_only,mean_only,248,-0.1152,-0.5105,0.2800,-0.5715,0.5677,NaN,NaN,0.0000


## 7. Incremental critical-minerals attribution

In [7]:
b=res[res.specification.eq('ff5_mom')].set_index(['panel','vintage','method']);a=res[res.specification.eq('ff5_mom_cm')].set_index(['panel','vintage','method']);inc=pd.DataFrame({'alpha_before_cm':b.daily_alpha,'alpha_after_cm':a.daily_alpha,'cm_beta':a.cm_beta,'cm_beta_hac_t':a.cm_beta_hac_t,'r2_before_cm':b.r2,'r2_after_cm':a.r2}).reset_index();inc['annualized_alpha_change']=252*(inc.alpha_after_cm-inc.alpha_before_cm);inc['incremental_r2']=inc.r2_after_cm-inc.r2_before_cm;inc.to_parquet(OUT/'critical_minerals_incremental_attribution.parquet',index=False);display(inc[inc.panel.eq('calendar_time')].round(4))

,panel,vintage,method,alpha_before_cm,alpha_after_cm,cm_beta,cm_beta_hac_t,r2_before_cm,r2_after_cm,annualized_alpha_change,incremental_r2
24,calendar_time,all_windows,consensus,-0.0008,-0.0004,0.4848,9.6169,0.3119,0.5144,0.0947,0.2024
25,calendar_time,all_windows,embedding,-0.0009,-0.0009,0.0214,0.9054,0.1174,0.1202,0.0042,0.0028
26,calendar_time,all_windows,hybrid,-0.0009,-0.0007,0.2143,8.5325,0.1073,0.3125,0.0419,0.2052
27,calendar_time,all_windows,hybrid_long_only,-0.0010,-0.0008,0.1885,10.2244,0.8195,0.8667,0.0368,0.0471
28,calendar_time,all_windows,llm,-0.0011,-0.0009,0.3019,13.0063,0.1885,0.4821,0.0590,0.2936
29,calendar_time,all_windows,matched_text,-0.0010,-0.0007,0.3745,6.7415,0.2331,0.4225,0.0732,0.1894


## 8. Frozen thematic-factor robustness

In [8]:
rr=[]
for pn,groups in [('vintage',panel.groupby(['vintage','method'])),('calendar_time',cal.groupby('method'))]:
 for key,z in groups:
  v,m=(key if pn=='vintage' else ('all_windows',key))
  for tf in ['broad_cm_equity_ex_market','producer_ex_market','base_metals_raw','materials_ex_market']:
   cols=['Mkt-RF','SMB','HML','RMW','CMA','Mom',tf];a=z[['dependent_return']+cols].dropna();r=sm.OLS(a.dependent_return,sm.add_constant(a[cols],has_constant='add')).fit(cov_type='HAC',cov_kwds={'maxlags':5,'use_correction':True});rr.append({'panel':pn,'vintage':v,'method':m,'theme_factor':tf,'days':len(a),'annualized_alpha':252*r.params['const'],'alpha_hac_t':r.tvalues['const'],'theme_beta':r.params[tf],'theme_beta_hac_t':r.tvalues[tf],'r2':r.rsquared})
rob=pd.DataFrame(rr);rob.to_parquet(OUT/'thematic_factor_robustness.parquet',index=False);display(rob[rob.panel.eq('calendar_time')].round(4))

,panel,vintage,method,theme_factor,days,annualized_alpha,alpha_hac_t,theme_beta,theme_beta_hac_t,r2
96,calendar_time,all_windows,consensus,broad_cm_equity_ex_market,248,-0.1019,-0.5534,0.4848,9.6169,0.5144
97,calendar_time,all_windows,consensus,producer_ex_market,248,-0.0801,-0.4506,0.4874,11.3280,0.5412
98,calendar_time,all_windows,consensus,base_metals_raw,248,-0.1597,-0.7957,0.5498,6.9271,0.4304
99,calendar_time,all_windows,consensus,materials_ex_market,248,-0.1635,-0.8155,0.4258,5.3613,0.4094
100,calendar_time,all_windows,embedding,broad_cm_equity_ex_market,248,-0.2157,-2.0005,0.0214,0.9054,0.1202
101,calendar_time,all_windows,embedding,producer_ex_market,248,-0.2127,-1.9890,0.0301,1.3927,0.1236
102,calendar_time,all_windows,embedding,base_metals_raw,248,-0.2170,-2.0192,0.0426,1.5604,0.1224
103,calendar_time,all_windows,embedding,materials_ex_market,248,-0.2193,-2.0372,0.0068,0.2001,0.1176
104,calendar_time,all_windows,hybrid,broad_cm_equity_ex_market,248,-0.1825,-1.8579,0.2143,8.5325,0.3125
105,calendar_time,all_windows,hybrid,producer_ex_market,248,-0.1725,-1.8866,0.2173,11.3178,0.3436


## 9. Gates and reproducibility manifest

In [9]:
p=res[(res.panel.eq('calendar_time'))&res.specification.eq('ff5_mom_cm')];gates=pd.DataFrame([{'gate':'six_frozen_methods','value':p.method.nunique(),'threshold':6,'passed':p.method.nunique()==6},{'gate':'minimum_calendar_days','value':p.days.min(),'threshold':230,'passed':p.days.min()>=230},{'gate':'four_vintages','value':res[res.panel.eq('vintage')].vintage.nunique(),'threshold':4,'passed':res[res.panel.eq('vintage')].vintage.nunique()==4},{'gate':'minimum_coverage','value':coverage.minimum_coverage.min(),'threshold':.90,'passed':coverage.minimum_coverage.min()>=.90},{'gate':'no_reselection','value':0,'threshold':0,'passed':True}]);gates.to_parquet(MAN/'decision_gates.parquet',index=False);display(gates)
arts=[dp,OUT/'daily_frozen_portfolio_returns.parquet',OUT/'daily_standard_factors.parquet',OUT/'daily_attribution_panel.parquet',OUT/'calendar_time_attribution_panel.parquet',OUT/'factor_attribution_results.parquet',OUT/'critical_minerals_incremental_attribution.parquet',OUT/'thematic_factor_robustness.parquet',MAN/'decision_gates.parquet'];pd.DataFrame([{'artifact':str(a.relative_to(ROOT)),'sha256':hashlib.sha256(a.read_bytes()).hexdigest()} for a in arts]).to_parquet(MAN/'artifact_inventory.parquet',index=False);(MAN/'manifest.json').write_text(json.dumps({'stage':DESIGN['version'],'design_sha256':dh,'methods':sorted(p.method.unique()),'calendar_days':int(p.days.min()),'all_gates_passed':bool(gates.passed.all()),'reselected':False},indent=2,sort_keys=True)+'\n');print('Saved:',OUT)

,gate,value,threshold,passed
0,six_frozen_methods,6.0,6.0,True
1,minimum_calendar_days,248.0,230.0,True
2,four_vintages,4.0,4.0,True
3,minimum_coverage,1.0,0.9,True
4,no_reselection,0.0,0.0,True


Saved: /Users/harmandeep/Desktop/Projects/Asset_Embeddings/geopolitical_exposure/sp500_critical_minerals/factor_attribution_v1


## Interpretation boundary

Critical-minerals beta measures sensitivity to an investable equity-theme factor, not proof of a physical supply-chain link. Alpha reduction after adding it indicates thematic attribution. Remaining alpha is exploratory—not proof of skill—given the short sample, absent transaction costs, and the fact that this attribution follows earlier pilot analysis. Robustness factors were frozen in 14A and cannot replace the primary factor based on preferred results.